In [6]:
import pandas as pd

# 1.Reading the 2 .csv file
sessions_df = pd.read_csv('../data/raw/sessions_data.csv')
web_df = pd.read_csv('../data/raw/cleaned_web_data.csv')

# 2. check if a (client_id + visit_id) pair reached confirm status
journey_status = (
    web_df.groupby(['client_id', 'visit_id'])['process_step']
    .apply(lambda steps: (steps == 'confirm').any())
    .reset_index(name='completed')
)

# 3. Merge with session_data.csv
sessions_df = pd.merge(sessions_df, journey_status, on=['client_id', 'visit_id'], how='left')

# with string for expandability- if we want to add more category 
sessions_df['journey_status'] = sessions_df['completed'].map({True: 'Completed', False: 'Incomplete'})

sessions_df.head()


,client_id,visit_id,step_count,completed,journey_status
0,555,637149525_38041617439_716659,5,True,Completed
1,647,40369564_40101682850_311847,5,True,Completed
2,934,7076463_57954418406_971348,4,False,Incomplete
3,1028,557292053_87239438319_391157,9,False,Incomplete
4,1104,543158812_46395476577_767725,1,False,Incomplete


In [7]:
print(sessions_df['completed'].value_counts())

print("=" * 60)
print('\n')

print(sessions_df['journey_status'].value_counts())

completed
True     37879
False    31568
Name: count, dtype: int64


journey_status
Completed     37879
Incomplete    31568
Name: count, dtype: int64


## Records that don’t follow the right order

In [8]:
# 2. Chronological ordering within a session
web_df['date_time'] = pd.to_datetime(web_df['date_time'])
web_df = web_df.sort_values(by=['client_id', 'visit_id', 'date_time']).reset_index(drop=True)

# 3. Numbering of steps
step_map = {'start': 0, 'step1': 1, 'step2': 2, 'step3': 3, 'confirm': 4}
web_df['step_num'] = web_df['process_step'].map(step_map)

# 4. The previous step within the same (client_id, visit_id) session
web_df['prev_step_num'] = web_df.groupby(['client_id', 'visit_id'])['step_num'].shift(1)
web_df['step_diff'] = web_df['step_num'] - web_df['prev_step_num']

# 5. Identifying various sequence deviations 
# for the first row if the prev_step_num is NaN = exclude

# A) Any irregular step (i.e., the step forward is not +1)
web_df['is_out_of_order'] = web_df['prev_step_num'].notna() & (web_df['step_diff'] != 1)

# B) Real step backward (e.g step_2 --> step_1, step_diff < 0)
web_df['is_step_back'] = web_df['prev_step_num'].notna() & (web_df['step_diff'] < 0)

# C) Repeat (step1 --> step1, step_diff == 0)
web_df['is_repeat'] = web_df['prev_step_num'].notna() & (web_df['step_diff'] == 0)

# D) Skip a step (step1 --> step3, step_diff > 1)
web_df['is_skip'] = web_df['prev_step_num'].notna() & (web_df['step_diff'] > 1)

# 6. Aggregation to the session level + joining to the df
order_summary = web_df.groupby(['client_id', 'visit_id']).agg(
    out_of_order_count=('is_out_of_order', 'sum'),
    step_backs_count=('is_step_back', 'sum'),
    repeats_count=('is_repeat', 'sum'),
    skips_count=('is_skip', 'sum'),
    starts_at_start=('step_num', lambda x: x.iloc[0] == 0) # Ellenőrzi, hogy 'start'-tal indult-e a session
).reset_index()

# Was there any sequencing error in the session? T/F
order_summary['follows_right_order'] = (order_summary['out_of_order_count'] == 0) & (order_summary['starts_at_start'])

# 7. Merge with the df
sessions_df = pd.merge(sessions_df, order_summary, on=['client_id', 'visit_id'], how='left')

# check
sessions_df.head()

,client_id,visit_id,step_count,completed,journey_status,out_of_order_count,step_backs_count,repeats_count,skips_count,starts_at_start,follows_right_order
0,555,637149525_38041617439_716659,5,True,Completed,1,0,0,0,True,False
1,647,40369564_40101682850_311847,5,True,Completed,1,0,0,0,True,False
2,934,7076463_57954418406_971348,4,False,Incomplete,3,0,3,0,True,False
3,1028,557292053_87239438319_391157,9,False,Incomplete,1,0,0,0,True,False
4,1104,543158812_46395476577_767725,1,False,Incomplete,0,0,0,0,True,True


**out_of_order_count** --> Total out-of-sequence occurrences (sum of the 3 types below):
Includes every instance where the next step was not the standard +1 progression (step_diff != 1). 
step_backs_count + repeats_count + skips_count

**step_backs_count** --> Step back

**repeats_count** --> Repetition

**skips_count** --> Skipped one (or more) steps

In [4]:
print(sessions_df['follows_right_order'].value_counts())

print("=" * 60)
print('\n')
print(sessions_df['starts_at_start'].value_counts())

follows_right_order
False    61321
True      8126
Name: count, dtype: int64


starts_at_start
True     63964
False     5483
Name: count, dtype: int64


starts_at_start == False

Possible reasons:

1. Process continuation in a different session
    a user initiates a process and then closes their browser, later reopen the interface (with a different visit_id), and the system resumes where they left off—meaning and second session begins directly at "step 2" or "step 3"
2. Sessions started before the experimental period (3/15/2017)
    a user initiated the process (the "start" action) before March 15 (activating the logging job) only the subsequent steps were recorded in the database.
3. Timeout, delete cookies, incognito mode
4. Direct link to the stepX (perhaps in an email or from a bookmark in the browser)

Questions // To do:
- check if there is a session with start but not confirm steps with different visit_id but same client_is (earlier timestamps)
- if a user didn't start at start but reached the confirmation stage, does that count as a completed process or not?
- when calculating step_duration or session_duration it is better to handle these cases separately or filter them out --> these sessions completion time would appear shorter than a regular session


- writing documentation (assumption log/readme, etc)
- add variants!!!
- statistics calculation - code from Marta:
============================================================
This is to see how many of each:
# We want to check how many sessions reached a specific step in the user journey
sessions_by_step = (timed_steps.drop_duplicates()
    .pivot_table(
        index="process_step",
        columns="variation",
        aggfunc="size"
    )
    .reindex(["start", "step_1", "step_2", "step_3", "confirm"])
)

display(sessions_by_step)

============================================================
And this to see the median of each:
# Check the median time for each step for users in group Test vs Control
median_step_times = (
    timed_steps.pivot_table(
        index="process_step",
        columns="variation",
        values="step_duration",
        aggfunc="median"
    )
    .reindex(
        index=["start", "step_1", "step_2", "step_3", "confirm"],
        columns=["Test", "Control"],
    )
)

display(median_step_times)

In [5]:
# save to csv
# session_df.to_csv('../data/raw/sessions_data_v2.csv', index=False)